# **SO SÁNH CÁC MODLE TRÊN TẬP MNIST**

Đánh giá toàn diện các mô hình học máy và học sâu đã được thực nghiệm qua 4 notebook chuyên đề:
* `1_Bayes`: Mô hình phân loại xác suất Naïve Bayes đa thức (`MultinomialNB-NP`).
* `2_Logistic`: Mô hình phân loại tuyến tính Hồi quy Logistic (`LogisticRegression-NP`).
* `3_MLP_CNN`: Mạng nơ-ron nhân tạo đa tầng (`MLP`) và Mạng nơ-ron tích chập (`CNN`).
* `4_Ensemble`: Phương pháp học kết hợp Boosting (`AdaBoost` với các bộ học cơ sở khác nhau).

Toàn bộ dữ liệu đánh giá được trích xuất trực tiếp từ các tệp kết quả chuẩn hóa trong thư mục `results/`, tập trung vào tập kiểm thử độc lập (Test Set) gồm 10.000 mẫu ảnh để đảm bảo tính khách quan và khả năng tổng quát hóa.



## **1. Giải thích các params tối ưu**

Trong phần này, kết quả của từng notebook được nạp riêng biệt. Với mỗi phương pháp, chúng ta trích xuất cấu hình siêu tham số tối ưu (`params`) trên tập kiểm thử, phân tích bản chất cơ chế hoạt động, ý nghĩa toán học và tác động thực tế của từng tham số đối với bài toán nhận dạng chữ số MNIST.


In [40]:
import os
import sys
import tempfile
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình thư mục tạm cho Matplotlib trên Windows
os.environ["MPLCONFIGDIR"] = tempfile.gettempdir()
warnings.filterwarnings("ignore")

In [41]:
# Xác định thư mục gốc dự án (PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / "pyproject.toml").is_file() and (PROJECT_ROOT / "src" / "pattern_recognition" / "config.py").is_file()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Không tìm thấy thư mục gốc dự án (chứa pyproject.toml).")
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Thiết lập môi trường thành công.")
print(f"Thư mục kết quả (RESULTS_DIR): {RESULTS_DIR}")
print(f"Thư mục biểu đồ (FIGURES_DIR): {FIGURES_DIR}")


Thiết lập môi trường thành công.
Thư mục kết quả (RESULTS_DIR): C:\Users\Lenovo\Downloads\Pattern-Recognition_1\results
Thư mục biểu đồ (FIGURES_DIR): C:\Users\Lenovo\Downloads\Pattern-Recognition_1\figures


### **1.1. Multinominal Naïve Bayes (từ notebook `1_Bayes`)**


In [42]:
df_bayes = pd.read_csv(RESULTS_DIR / "1_Bayes_metrics.csv")
test_bayes = df_bayes[df_bayes["split"] == "test"].copy()

print("══ KẾT QUẢ BAYES — TẬP TEST ══")
for _, row in test_bayes.iterrows():
    latency_us = row["predict_seconds"] / row["n_samples"] * 1e6

    print(f"\n {row['model']}")
    print(f"  Cấu hình: {row['params']}")
    print("─" * 50)
    print(f"  {'Accuracy':<28} {row['accuracy']*100:>10.2f} %")
    print(f"  {'Macro-F1':<28} {row['f1_macro']*100:>10.2f} %")
    print(f"  {'Thời gian huấn luyện':<28} {row['train_seconds']:>10.4f} s")
    print(f"  {'Thời gian dự đoán':<28} {row['predict_seconds']:>10.4f} s")
    print(f"  {'Số mẫu dự đoán':<28} {row['n_samples']:>10,.0f}")
    print(f"  {'Độ trễ trung bình / mẫu':<28} {latency_us:>10.1f} µs")
    print("─" * 50)

══ KẾT QUẢ BAYES — TẬP TEST ══

 MultinomialNB-NP
  Cấu hình: alpha=0.0001
──────────────────────────────────────────────────
  Accuracy                          83.56 %
  Macro-F1                          83.41 %
  Thời gian huấn luyện             0.0541 s
  Thời gian dự đoán                0.2689 s
  Số mẫu dự đoán                   10,000
  Độ trễ trung bình / mẫu            26.9 µs
──────────────────────────────────────────────────


#### **Nhận xét kết quả mô hình `MultinomialNB-NP`**

* **Accuracy = 83.56%:** Mô hình dự đoán đúng **8,356/10,000 ảnh**, còn **1,644 ảnh bị nhận diện sai**. Kết quả cho thấy mô hình học được những đặc điểm cơ bản của chữ số, nhưng tỷ lệ lỗi **16.44%** vẫn đáng kể. Đây là mức kết quả phù hợp để làm baseline.

* **Macro-F1 = 83.41%:** Macro-F1 thấp hơn Accuracy chỉ **0.15 điểm phần trăm**, cho thấy kết quả tổng thể khá nhất quán giữa hai cách đánh giá. Tuy nhiên, cần xem ma trận nhầm lẫn để xác định các chữ số thường bị nhận diện sai.

* **Tốc độ xử lý:** Thời gian huấn luyện **0.0595 giây** do mô hình chủ yếu tổng hợp thống kê từ dữ liệu, không cần tối ưu qua nhiều vòng lặp. Mô hình dự đoán **10,000 ảnh trong 0.3330 giây**, tương đương **33.3 µs/ảnh**. Chi phí tính toán thấp giúp mô hình phù hợp với các thử nghiệm nhanh và việc phân loại lượng lớn ảnh.

* **Param: `alpha = 0.0001`:** Hệ số làm trơn nhỏ giúp tránh xác suất bằng 0 tại những vị trí điểm ảnh chưa xuất hiện nét chữ trong một lớp, đồng thời ít làm thay đổi phân bố đã học. Đây là tham số hỗ trợ tính toán xác suất; kết quả hiện tại chưa chứng minh giá trị này là tối ưu theo thực nghiệm.

---
### **1.2. Khảo sát mô hình Hồi quy Logistic (từ notebook `2_Logistic`)**


In [43]:
df_logistic = pd.read_csv(RESULTS_DIR / "2_Logistic_metrics.csv")
test_logistic = df_logistic[df_logistic["split"] == "test"].copy()

print("══ KẾT QUẢ LOGISTIC REGRESSION — TẬP TEST ══")
for _, row in test_logistic.iterrows():
    latency_us = row["predict_seconds"] / row["n_samples"] * 1e6

    print(f"\n {row['model']}")
    print(f"  Cấu hình: {row['params']}")
    print("─" * 50)
    print(f"  {'Accuracy':<28} {row['accuracy']*100:>10.2f} %")
    print(f"  {'Macro-F1':<28} {row['f1_macro']*100:>10.2f} %")
    print(f"  {'Thời gian huấn luyện':<28} {row['train_seconds']:>10.4f} s")
    print(f"  {'Thời gian dự đoán':<28} {row['predict_seconds']:>10.4f} s")
    print(f"  {'Số mẫu dự đoán':<28} {row['n_samples']:>10,.0f}")
    print(f"  {'Độ trễ trung bình / mẫu':<28} {latency_us:>10.1f} µs")
    print("─" * 50)

══ KẾT QUẢ LOGISTIC REGRESSION — TẬP TEST ══

 LogisticRegression-NP
  Cấu hình: eta=0.5,max_epochs=30,lam=1e-4,strategy=OneVsRest
──────────────────────────────────────────────────
  Accuracy                          86.84 %
  Macro-F1                          86.55 %
  Thời gian huấn luyện            10.7028 s
  Thời gian dự đoán                0.0288 s
  Số mẫu dự đoán                   10,000
  Độ trễ trung bình / mẫu             2.9 µs
──────────────────────────────────────────────────


#### **Nhận xét kết quả mô hình `LogisticRegression`**

* **Accuracy = 86.84%:** Mô hình dự đoán đúng **8,684/10,000 ảnh**, còn **1,316 ảnh bị nhận diện sai**. So với `MultinomialNB` (Accuracy = **83.56%**), độ chính xác tăng **3.28 điểm phần trăm**, tương ứng giảm **328 dự đoán sai** trên cùng tập kiểm tra. Macro-F1 đạt **86.55%**, cho thấy hiệu quả phân loại trung bình giữa các lớp tương đối đồng đều.

* **Giới hạn mô hình:** Tỷ lệ lỗi còn **13.16%**. Hồi quy Logistic xây dựng ranh giới phân loại tuyến tính nên có thể hạn chế khả năng biểu diễn các quan hệ phức tạp giữa những nét chữ, nhất là với các chữ số có hình dạng gần giống nhau.

* **Các siêu tham số:**
  + `eta = 0.5`: Tốc độ học quyết định độ lớn của mỗi bước cập nhật trọng số. Giá trị này ảnh hưởng đến tốc độ và độ ổn định của quá trình học; cần xem đường cong loss để đánh giá thêm.

  + `max_epochs = 30`: Mô hình được huấn luyện tối đa **30 lượt duyệt toàn bộ dữ liệu**. Đây là giới hạn số vòng lặp, chưa đủ để kết luận mô hình đã hội tụ.

  + `lam = 1e-4`: Hệ số điều chuẩn L2 giúp hạn chế trọng số quá lớn và giảm nguy cơ quá khớp. Mức độ phù hợp của hệ số này nên được đánh giá trên tập validation.

  + `strategy = OneVsRest`: Mô hình huấn luyện **10 bộ phân loại nhị phân**, mỗi bộ phân biệt một chữ số với các chữ số còn lại. Khi dự đoán, lớp có đầu ra sigmoid cao nhất được chọn.

* **Thời gian thực thi:** Huấn luyện mất **10.7028 giây**. Dự đoán **10,000 mẫu** mất **0.0288 giây**, tương đương độ trễ trung bình khoảng **2.9 µs/mẫu**.

---
### **1.3. Khảo sát mô hình Mạng Nơ-ron Nhân tạo và Mạng Tích chập (từ notebook `3_MLP_CNN`)**


In [44]:
df_mlp_cnn = pd.read_csv(RESULTS_DIR / "3_MLP_CNN_metrics.csv")
test_mlp_cnn = df_mlp_cnn[df_mlp_cnn["split"] == "test"].copy()

print("══ KẾT QUẢ MLP & CNN — TẬP TEST ══")
for _, row in test_mlp_cnn.iterrows():
    latency_us = row["predict_seconds"] / row["n_samples"] * 1e6

    print(f"\n {row['model']}")
    print(f"  Cấu hình: {row['params']}")
    print("─" * 50)
    print(f"  {'Accuracy':<28} {row['accuracy']*100:>10.2f} %")
    print(f"  {'Macro-F1':<28} {row['f1_macro']*100:>10.2f} %")
    print(f"  {'Thời gian huấn luyện':<28} {row['train_seconds']:>10.4f} s")
    print(f"  {'Thời gian dự đoán':<28} {row['predict_seconds']:>10.4f} s")
    print(f"  {'Số mẫu dự đoán':<28} {row['n_samples']:>10,.0f}")
    print(f"  {'Độ trễ trung bình / mẫu':<28} {latency_us:>10.1f} µs")
    print("─" * 50)

══ KẾT QUẢ MLP & CNN — TẬP TEST ══

 MLP
  Cấu hình: arch=784-256-128-10,activation=ReLU,init=He,lr=0.1,momentum=0.5,batch=64,weight_decay=1e-4,best_epoch=18
──────────────────────────────────────────────────
  Accuracy                          98.26 %
  Macro-F1                          98.25 %
  Thời gian huấn luyện            33.5715 s
  Thời gian dự đoán                0.0216 s
  Số mẫu dự đoán                   10,000
  Độ trễ trung bình / mẫu             2.2 µs
──────────────────────────────────────────────────

 CNN
  Cấu hình: conv=32x4x4,pool=2x2,dense=128,optimizer=Adam,lr=1e-3,batch=128,max_epochs=10,patience=2,best_epoch=7
──────────────────────────────────────────────────
  Accuracy                          98.76 %
  Macro-F1                          98.76 %
  Thời gian huấn luyện            51.6119 s
  Thời gian dự đoán                0.4285 s
  Số mẫu dự đoán                   10,000
  Độ trễ trung bình / mẫu            42.9 µs
───────────────────────────────────────────

#### **Nhận xét kết quả mô hình `MLP`**

* **Accuracy = 98.26%:** Mô hình dự đoán đúng **9,826/10,000 ảnh**, còn **174 ảnh bị nhận diện sai**. So với `LogisticRegression`, độ chính xác tăng **8.08 điểm phần trăm**, tương ứng giảm **808 lỗi** trên cùng tập kiểm tra. Mức cải thiện này cho thấy các tầng phi tuyến giúp nhận dạng chữ viết hiệu quả hơn rõ rệt so với mô hình tuyến tính.

* **Macro-F1 = 98.25%:** Chỉ thấp hơn Accuracy **0.01 điểm phần trăm**, cho thấy chất lượng tổng thể khá nhất quán giữa hai cách đánh giá. Mô hình đạt kết quả cao cả về tỷ lệ dự đoán đúng lẫn F1 trung bình theo lớp; tuy nhiên, cần xem kết quả từng lớp để xác định những chữ số còn khó nhận diện.

* **Tốc độ xử lý:** Huấn luyện mất **27.3434 giây**, trong khi dự đoán toàn bộ **10,000 ảnh chỉ mất 0.0206 giây**, tương đương **2.1 µs/ảnh**. Trong phép đo này, MLP dự đoán nhanh hơn CNN khoảng **19.9 lần**, dù độ chính xác chỉ thấp hơn **0.50 điểm phần trăm**. Đây là lợi thế đáng chú ý khi cần xử lý nhiều ảnh và ưu tiên tốc độ suy luận. Độ trễ trên mỗi ảnh ở đây là giá trị trung bình khi xử lý theo lô.

* **Các siêu tham số:**

  + `arch = 784-256-128-10`, `activation = ReLU`, `init = He`: Hai tầng ẩn kết hợp ReLU cho phép học những tổ hợp đặc trưng phức tạp hơn Logistic Regression; khởi tạo He hỗ trợ quá trình huấn luyện mạng dùng ReLU.

  + `lr = 0.1`, `momentum = 0.5`, `batch = 64`, `weight_decay = 1e-4`: Cấu hình cập nhật theo lô có momentum và điều chuẩn trọng số đạt kết quả test cao trong thời gian huấn luyện đã ghi nhận. Chưa có thử nghiệm đối chiếu để tách riêng mức đóng góp của từng tham số.

  + `best_epoch = 18`: Checkpoint tốt nhất theo tiêu chí validation nằm ở **epoch 18**. Đây là mốc chọn trọng số để đánh giá, không phải bằng chứng rằng mô hình dừng huấn luyện tại epoch này.

#### **Nhận xét kết quả mô hình `CNN`**

* **Accuracy = 98.76%:** Mô hình dự đoán đúng **9,876/10,000 ảnh**, còn **124 ảnh bị nhận diện sai**. So với MLP, độ chính xác tăng **0.50 điểm phần trăm**, tương ứng giảm **50 lỗi**, tức khoảng **28.7% số lỗi của MLP**. Vì cả hai mô hình đều đạt độ chính xác cao, mức tăng nhỏ về Accuracy vẫn mang lại mức giảm lỗi đáng kể.

* **Macro-F1 = 98.76%:** Bằng Accuracy ở độ chính xác hiển thị và cao hơn MLP **0.51 điểm phần trăm**. Như vậy, CNN cải thiện cả tỷ lệ dự đoán đúng và F1 trung bình theo lớp. Đây là mô hình có chất lượng nhận dạng tốt nhất trong các kết quả đã cung cấp.

* **Tốc độ xử lý:** Huấn luyện mất **42.7136 giây**, lâu hơn MLP khoảng **1.56 lần**. Dự đoán **10,000 ảnh mất 0.4099 giây**, tương đương **41.0 µs/ảnh**, chậm hơn MLP khoảng **19.9 lần** trong phép đo theo lô. CNN vì vậy đánh đổi chi phí tính toán cao hơn để giảm số lỗi nhận dạng. Nếu ưu tiên độ chính xác, CNN có lợi thế; nếu ưu tiên tốc độ dự đoán, MLP cho sự cân bằng tốt hơn trong thử nghiệm này.

* **Các siêu tham số:**

  + `conv = 32x4x4`, `pool = 2x2`, `dense = 128`: Cấu trúc tích chập khai thác các nét chữ trong không gian hai chiều, cung cấp một cơ sở hợp lý để giải thích lợi thế nhận dạng so với MLP. Tuy nhiên, kết quả tổng hợp chưa xác định riêng đóng góp của từng thành phần.

  + `optimizer = Adam`, `lr = 1e-3`, `batch = 128`: Cấu hình tối ưu này đạt kết quả test **98.76%**. Batch lớn hơn MLP nhưng thời gian dự đoán vẫn cao hơn, cho thấy kích thước lô không quyết định tốc độ khi kiến trúc và lượng tính toán khác nhau.

  + `max_epochs = 10`, `patience = 2`, `best_epoch = 7`: Checkpoint tốt nhất theo tiêu chí validation xuất hiện ở **epoch 7**. Dù mốc này sớm hơn MLP, tổng thời gian huấn luyện CNN vẫn dài hơn; vì vậy, không thể dùng số epoch đơn thuần để so sánh tốc độ của hai mô hình.

---
### **1.4. Khảo sát phương pháp Học kết hợp Boosting (từ notebook `4_Ensemble`)**


In [45]:
df_ensemble = pd.read_csv(RESULTS_DIR / "4_Ensemble_metrics.csv")
test_ensemble = df_ensemble[df_ensemble["split"] == "test"].copy()

print("══ KẾT QUẢ ENSEMBLE — TẬP TEST ══")
for _, row in test_ensemble.iterrows():
    latency_us = row["predict_seconds"] / row["n_samples"] * 1e6

    print(f"\n {row['model']}")
    print(f"  Cấu hình: {row['params']}")
    print("─" * 50)
    print(f"  {'Accuracy':<28} {row['accuracy']*100:>10.2f} %")
    print(f"  {'Macro-F1':<28} {row['f1_macro']*100:>10.2f} %")
    print(f"  {'Thời gian huấn luyện':<28} {row['train_seconds']:>10.2f} s")
    print(f"  {'Thời gian dự đoán':<28} {row['predict_seconds']:>10.4f} s")
    print(f"  {'Số mẫu dự đoán':<28} {row['n_samples']:>10,.0f}")
    print(f"  {'Độ trễ trung bình / mẫu':<28} {latency_us:>10.1f} µs")
    print("─" * 50)

══ KẾT QUẢ ENSEMBLE — TẬP TEST ══

 AdaBoost (depth=3)
  Cấu hình: n_estimators=200,learning_rate=0.5,estimator=DecisionTreeClassifier(max_depth=3)
──────────────────────────────────────────────────
  Accuracy                          91.75 %
  Macro-F1                          91.70 %
  Thời gian huấn luyện           11019.24 s
  Thời gian dự đoán                2.6681 s
  Số mẫu dự đoán                   10,000
  Độ trễ trung bình / mẫu           266.8 µs
──────────────────────────────────────────────────

 AdaBoost (LogReg)
  Cấu hình: n_estimators=50,learning_rate=0.5,estimator=LogisticRegression(max_iter=500)
──────────────────────────────────────────────────
  Accuracy                          85.06 %
  Macro-F1                          84.85 %
  Thời gian huấn luyện              48.39 s
  Thời gian dự đoán                0.3013 s
  Số mẫu dự đoán                   10,000
  Độ trễ trung bình / mẫu            30.1 µs
──────────────────────────────────────────────────

 AdaBoost (s

#### **Nhận xét kết quả mô hình `AdaBoost (depth=3)`**

* **Accuracy = 91.75%:** Mô hình dự đoán đúng **9,175/10,000 ảnh**, còn **825 ảnh bị nhận diện sai**, đạt kết quả cao nhất trong bốn biến thể AdaBoost. So với Logistic Regression đơn lẻ, độ chính xác tăng **1.57 điểm phần trăm**, tương ứng giảm **157 lỗi**. Tuy nhiên, kết quả vẫn thấp hơn đáng kể so với MLP và CNN.

* **Macro-F1 = 91.70%:** Chỉ thấp hơn Accuracy **0.05 điểm phần trăm**, cho thấy hai cách đánh giá phản ánh chất lượng tổng thể khá nhất quán. Đây cũng là Macro-F1 cao nhất trong nhóm AdaBoost, củng cố lợi thế của cấu hình này về chất lượng nhận dạng.

* **Tốc độ xử lý:** Thời gian huấn luyện lên tới **11,019.24 giây**, tương đương khoảng **3.06 giờ**. Dự đoán **10,000 ảnh mất 2.6681 giây**, trung bình **266.8 µs/ảnh**. So với AdaBoost stump, mô hình tăng **13.22 điểm phần trăm Accuracy**, nhưng thời gian huấn luyện dài hơn khoảng **45.5 lần**. Mức cải thiện rõ rệt đi kèm chi phí huấn luyện rất lớn.

* **Các siêu tham số:**

  + `n_estimators = 200`, `learning_rate = 0.5`: Cấu hình cho phép tối đa 200 bộ học được xây dựng tuần tự, góp phần làm tăng chi phí huấn luyện.

  + `estimator = DecisionTreeClassifier(max_depth=3)`: Cây sâu 3 có thể kết hợp nhiều điều kiện trên các điểm ảnh, tạo khả năng phân biệt phong phú hơn stump. Tuy nhiên, hai cấu hình còn khác số lượng cây nên chưa thể quy toàn bộ mức cải thiện cho độ sâu.


#### **Nhận xét kết quả mô hình `AdaBoost (LogReg)`**

* **Accuracy = 85.06%:** Mô hình dự đoán đúng **8,506/10,000 ảnh**, còn **1,494 ảnh bị nhận diện sai**. So với Logistic Regression đơn lẻ đã báo cáo, độ chính xác giảm **5.12 điểm phần trăm**, tương ứng tăng **512 lỗi**. Kết quả cho thấy cấu hình boosting này chưa mang lại lợi ích về độ chính xác so với mô hình đơn lẻ.

* **Macro-F1 = 84.85%:** Thấp hơn Accuracy **0.21 điểm phần trăm** và đứng thứ hai trong bốn biến thể AdaBoost. Cả hai chỉ số đều thấp hơn mức của Logistic Regression đơn lẻ, cho thấy chất lượng tổng thể giảm theo cả hai cách đánh giá.

* **Tốc độ xử lý:** Huấn luyện mất **48.39 giây**, gần với GaussianNB và thấp hơn nhiều so với hai cấu hình dùng cây. Dự đoán **10,000 ảnh mất 0.3013 giây**, tương đương **30.1 µs/ảnh**, nhanh nhất trong nhóm AdaBoost. Dù có lợi thế về tốc độ, cấu hình này vẫn cần được cân nhắc vì độ chính xác thấp hơn Logistic Regression đơn lẻ.

* **Các siêu tham số:**

  + `n_estimators = 50`, `learning_rate = 0.5`: Cấu hình sử dụng tối đa 50 bộ học. Kết quả cho thấy việc kết hợp nhiều mô hình không tự động bảo đảm chất lượng tốt hơn.

  + `estimator = LogisticRegression(max_iter=500)`: Mỗi bộ học cơ sở vẫn có ranh giới tuyến tính. Việc tái phân bổ trọng số mẫu có thể ảnh hưởng đến kết quả, nhưng output hiện tại chưa đủ để xác định nguyên nhân cụ thể của mức giảm Accuracy.


#### **Nhận xét kết quả mô hình `AdaBoost (stump)`**

* **Accuracy = 78.53%:** Mô hình dự đoán đúng **7,853/10,000 ảnh**, còn **2,147 ảnh bị nhận diện sai**. Kết quả thấp hơn `MultinomialNB` **5.03 điểm phần trăm** và thấp hơn AdaBoost depth=3 **13.22 điểm phần trăm**. Như vậy, cấu hình hiện tại chưa tạo được bộ phân loại đủ hiệu quả cho bài toán nhận dạng chữ số.

* **Macro-F1 = 78.37%:** Thấp hơn Accuracy **0.16 điểm phần trăm**. Hai chỉ số đều ở khoảng 78%, cho thấy hạn chế xuất hiện cả ở tỷ lệ dự đoán đúng và F1 trung bình theo lớp.

* **Tốc độ xử lý:** Huấn luyện mất **241.97 giây**, khoảng **4.03 phút**. Dự đoán **10,000 ảnh mất 2.4197 giây**, tương đương **242.0 µs/ảnh**. So với AdaBoost depth=3, thời gian huấn luyện giảm mạnh nhưng thời gian dự đoán chỉ giảm khoảng **9.3%**, trong khi độ chính xác giảm đáng kể. Cấu hình này cũng kém AdaBoost LogReg cả về độ chính xác lẫn thời gian xử lý trong thử nghiệm.

* **Các siêu tham số:**

  + `n_estimators = 100`, `learning_rate = 0.5`: Với cấu hình tối đa 100 bộ học, tỷ lệ lỗi vẫn còn **21.47%**. Kết quả chưa cho thấy số lượng bộ học hiện tại đủ bù đắp hạn chế của từng stump.

  + `estimator = DecisionTreeClassifier(max_depth=1)`: Mỗi cây chỉ phân chia theo một đặc trưng tại một ngưỡng. Khả năng biểu diễn đơn giản này là một lý do hợp lý khiến mô hình khó phân biệt các cấu trúc nét chữ phức tạp.


#### **Nhận xét kết quả mô hình `AdaBoost (GaussianNB)`**

* **Accuracy = 56.41%:** Mô hình dự đoán đúng **5,641/10,000 ảnh**, còn **4,359 ảnh bị nhận diện sai**, thấp nhất trong bốn biến thể AdaBoost. Tỷ lệ lỗi **43.59%** cho thấy cấu hình này chưa đáp ứng tốt bài toán nhận dạng chữ số.

* **Macro-F1 = 50.83%:** Thấp hơn Accuracy tới **5.58 điểm phần trăm**, là khoảng cách lớn nhất trong nhóm. Khi đánh giá F1 với trọng số ngang nhau cho từng lớp, kết quả còn thấp hơn rõ rệt so với tỷ lệ dự đoán đúng tổng thể. Cần kiểm tra F1 theo lớp và ma trận nhầm lẫn để xác định những chữ số có chất lượng nhận dạng thấp.

* **Tốc độ xử lý:** Huấn luyện mất **46.76 giây**, nhanh nhất nhóm nhưng chỉ ít hơn AdaBoost LogReg **1.63 giây**. Ngược lại, dự đoán **10,000 ảnh mất 7.8411 giây**, tương đương **784.1 µs/ảnh**, chậm nhất nhóm và khoảng **26 lần** thời gian của AdaBoost LogReg. Lợi thế nhỏ về thời gian huấn luyện không bù được độ chính xác thấp và chi phí dự đoán cao.

* **Các siêu tham số:**

  + `n_estimators = 50`, `learning_rate = 0.5`: Cùng hai giá trị cấu hình này với AdaBoost LogReg, nhưng Accuracy thấp hơn **28.65 điểm phần trăm**. Chênh lệch cho thấy lựa chọn bộ học cơ sở ảnh hưởng lớn đến hiệu quả của ensemble trong thử nghiệm.

  + `estimator = GaussianNB()`: Giả định phân phối Gaussian cho từng đặc trưng có thể chưa phù hợp với phân bố độ sáng điểm ảnh. Đây là hướng giải thích cần kiểm chứng; riêng output chưa đủ để kết luận việc gán trọng số mẫu đã làm sai lệch các ước lượng của mô hình.


### **1.5.Xuất `overall_comparison_metrics.csv`**

Toàn bộ các tệp kết quả riêng lẻ được hợp nhất thành file duy nhất và xuất bản ra tệp `results/overall_comparison_metrics.csv`. Sau đó, chúng ta xây dựng bảng tổng hợp xếp hạng tất cả các mô hình trên tập kiểm thử độc lập (Test Set, 10.000 mẫu) theo thứ tự độ chính xác giảm dần.


In [46]:
# Hợp nhất dữ liệu từ 4 file CSV
df_all = pd.concat([df_bayes, df_logistic, df_mlp_cnn, df_ensemble], ignore_index=True)

# Xuất tệp dữ liệu tổng hợp
output_metrics_file = RESULTS_DIR / "overall_comparison_metrics.csv"
df_all.to_csv(output_metrics_file, index=False)

print(f"Đã hợp nhất thành công {len(df_all)} dòng dữ liệu từ 4 notebook.")
print(f"Đã lưu tệp tổng hợp tại: {output_metrics_file}")


Đã hợp nhất thành công 24 dòng dữ liệu từ 4 notebook.
Đã lưu tệp tổng hợp tại: C:\Users\Lenovo\Downloads\Pattern-Recognition_1\results\overall_comparison_metrics.csv


In [47]:
# Lọc tập kiểm thử (test set) và sắp xếp theo Accuracy giảm dần
test_df = df_all[df_all["split"] == "test"].copy()
test_df = test_df.sort_values("accuracy", ascending=False).reset_index(drop=True)

# Hàm xác định notebook nguồn cho từng mô hình
def get_source_notebook(model_name):
    if "MultinomialNB" in model_name:
        return "1_Bayes"
    elif "LogisticRegression" in model_name:
        return "2_Logistic"
    elif model_name in ["MLP", "CNN"]:
        return "3_MLP_CNN"
    else:
        return "4_Ensemble"

test_df["notebook_source"] = test_df["model"].apply(get_source_notebook)

# Xây dựng bảng hiển thị chuẩn mực
display_table = pd.DataFrame({
    "Notebook nguồn": test_df["notebook_source"],
    "Mô hình": test_df["model"],
    "Độ chính xác (Acc %)": test_df["accuracy"].apply(lambda x: f"{x*100:.2f}%"),
    "Macro-F1 (%)": test_df["f1_macro"].apply(lambda x: f"{x*100:.2f}%"),
    "Precision (%)": test_df["precision_macro"].apply(lambda x: f"{x*100:.2f}%"),
    "Recall (%)": test_df["recall_macro"].apply(lambda x: f"{x*100:.2f}%"),
    "Thời gian Train (s)": test_df["train_seconds"].apply(lambda x: f"{x:.2f}"),
    "Thời gian Test (s)": test_df["predict_seconds"].apply(lambda x: f"{x:.4f}"),
    "Độ trễ / mẫu (µs)": (test_df["predict_seconds"] / test_df["n_samples"] * 1e6).apply(lambda x: f"{x:5.1f}"),
    "Tham số cấu hình chính": test_df["params"]
})

print("=== Bảng so sánh kết quả trên tập Test (MNIST TEST SET: 10.000 MẪU) ===")
display(display_table)


=== Bảng so sánh kết quả trên tập Test (MNIST TEST SET: 10.000 MẪU) ===


,Notebook nguồn,Mô hình,Độ chính xác (Acc %),Macro-F1 (%),Precision (%),Recall (%),Thời gian Train (s),Thời gian Test (s),Độ trễ / mẫu (µs),Tham số cấu hình chính
0,3_MLP_CNN,CNN,98.76%,98.76%,98.77%,98.75%,51.61,0.4285,42.9,"conv=32x4x4,pool=2x2,dense=128,optimizer=Adam,..."
1,3_MLP_CNN,MLP,98.26%,98.25%,98.25%,98.24%,33.57,0.0216,2.2,"arch=784-256-128-10,activation=ReLU,init=He,lr..."
2,4_Ensemble,AdaBoost (depth=3),91.75%,91.70%,91.81%,91.70%,11019.24,2.6681,266.8,"n_estimators=200,learning_rate=0.5,estimator=D..."
3,2_Logistic,LogisticRegression-NP,86.84%,86.55%,86.87%,86.58%,10.70,0.0288,2.9,"eta=0.5,max_epochs=30,lam=1e-4,strategy=OneVsRest"
4,4_Ensemble,AdaBoost (LogReg),85.06%,84.85%,85.11%,84.79%,48.39,0.3013,30.1,"n_estimators=50,learning_rate=0.5,estimator=Lo..."
5,1_Bayes,MultinomialNB-NP,83.56%,83.41%,84.24%,83.25%,0.05,0.2689,26.9,alpha=0.0001
6,4_Ensemble,AdaBoost (stump),78.53%,78.37%,78.80%,78.26%,241.97,2.4197,242.0,"n_estimators=100,learning_rate=0.5,estimator=D..."
7,4_Ensemble,AdaBoost (GaussianNB),56.41%,50.83%,71.58%,55.63%,46.76,7.8411,784.1,"n_estimators=50,learning_rate=0.5,estimator=Ga..."


## 2. **So sánh tổng thể các mô hình**

### **2.1. Công bằng trong thực nghiệm so sánh các mô hình**

Các mô hình không cần có cùng số epoch hoặc vòng lặp. Tính công bằng phụ thuộc vào dữ liệu, quy trình chọn cấu hình và điều kiện đánh giá.

#### **2.1.1. Số epoch khác nhau có ảnh hưởng đến tính công bằng không?**

Logistic Regression đặt `max_epochs = 30`, còn MLP và CNN có `best_epoch` lần lượt là **18** và **7**. Đây là hai loại thông tin khác nhau: giới hạn huấn luyện và epoch có trọng số được chọn. Ngoài ra, Logistic Regression cập nhật theo toàn bộ dữ liệu, còn MLP/CNN cập nhật nhiều lần theo mini-batch trong mỗi epoch. Vì vậy, không thể dựa vào số epoch để kết luận mô hình nào được ưu tiên huấn luyện hơn.

#### **2.1.2. Patience khác nhau giữa MLP và CNN có hợp lý không?**

Có thể sử dụng patience khác nhau nếu lựa chọn dựa trên validation và không điều chỉnh theo kết quả test.

* **MLP — `patience = 4`:** Validation loss tăng ở epoch 8–9 rồi giảm trở lại tại epoch 10. Trên chuỗi kết quả này, patience bằng 2 sẽ dừng quá sớm. MLP thực tế chạy **20 epoch**, sau đó lấy trọng số tại **epoch 18**, đạt Accuracy **98.26%**.

* **CNN — `patience = 2`:** Validation loss ở epoch 7–9 lần lượt là **0.0461 → 0.0462 → 0.0520**. Mô hình dừng tại **epoch 9**, lấy trọng số tại **epoch 7**, đạt Accuracy **98.76%**.

Các output giải thích được quy tắc dừng đã dùng, nhưng chưa chứng minh những giá trị patience này là tối ưu.

#### **2.1.3. Vì sao AdaBoost có nhiều vòng lặp nhưng hiệu quả còn hạn chế?**

Một vòng AdaBoost xây dựng thêm bộ học cơ sở, không tương đương một epoch của mạng nơ-ron. Số vòng lớn hơn không bảo đảm chất lượng tốt hơn.

AdaBoost depth=3 đặt tối đa **200 cây**, đạt **91.75%**, mất **11,019.24 giây**, khoảng **3.06 giờ** để huấn luyện. Kết quả cao hơn Logistic Regression (**90.18%**) nhưng thấp hơn MLP (**98.26%**) và CNN (**98.76%**). Vì vậy, không nên gọi toàn bộ AdaBoost là “kém nhất”; hạn chế nổi bật là chi phí lớn so với chất lượng đạt được ở cấu hình này.

### **2.2. Phân tích kết quả thực nghiệm**

Kết quả được đánh giá trên cùng 10.000 mẫu ảnh kiểm thử, tập trung vào ba tiêu chí: khả năng phân loại, thời gian huấn luyện và độ trễ suy diễn.

#### **2.2.1. Khả năng phân loại**

- **CNN dẫn đầu** với Accuracy và Macro-F1 cùng đạt **98,76%**; MLP theo sát với lần lượt **98,26%** và **98,25%**. CNN chỉ hơn MLP **0,50 điểm phần trăm**.
- Trong nhóm mô hình truyền thống, **AdaBoost (depth=3)** đạt kết quả cao nhất (**91,75% Accuracy; 91,70% Macro-F1**), tiếp theo là **LogisticRegression-NP** (**90,18%; 90,03%**).
- Các mô hình còn lại có kết quả thấp hơn: AdaBoost (LogReg) đạt **85,06%**, MultinomialNB-NP **83,56%**, AdaBoost (stump) **78,53%** và AdaBoost (GaussianNB) **56,41%**.

#### **2.2.2. Thời gian huấn luyện**

- **MultinomialNB-NP nhanh nhất**, chỉ mất **0,06 giây**.
- MLP và CNN lần lượt mất **30,27 giây** và **43,59 giây**, trong khi LogisticRegression-NP mất **32,29 giây**.
- **AdaBoost (depth=3) tốn nhiều thời gian nhất**: **11.019,24 giây** (khoảng 3,06 giờ), nhưng Accuracy vẫn thấp hơn CNN **7,01 điểm phần trăm**.

#### **2.2.3. Độ trễ từng mẫu**

- **MLP và LogisticRegression-NP nhanh nhất**, lần lượt đạt **2,3 µs/mẫu** và **2,6 µs/mẫu**.
- MultinomialNB-NP mất **33,3 µs/mẫu**; CNN mất **41,3 µs/mẫu**, vẫn phù hợp với các ứng dụng cần phản hồi nhanh.
- Các mô hình boosting có độ trễ cao hơn: AdaBoost (depth=3) mất **266,8 µs/mẫu**, còn AdaBoost (GaussianNB) mất **784,1 µs/mẫu**.

**Nhận xét chung:** CNN phù hợp khi ưu tiên độ chính xác; MLP nổi bật với độ chính xác cao cùng tốc độ suy diễn nhanh. MultinomialNB-NP có lợi thế về thời gian huấn luyện, trong khi AdaBoost (depth=3) có chi phí huấn luyện lớn so với kết quả đạt được.

## **3.Trực quan hóa so sánh các mô hình**

Nhằm cung cấp cái nhìn trực quan toàn diện, chúng ta xây dựng hệ thống 3 biểu đồ chuẩn mực được lưu trữ trực tiếp vào thư mục `figures/`:
1. `Comp_01_accuracy_f1_comparison.png`: Biểu đồ cột nhóm so sánh Accuracy và Macro-F1 của tất cả các mô hình.

In [53]:
model_names = test_df["model"].astype(str).tolist()
acc_scores = (test_df["accuracy"] * 100).tolist()
f1_scores = (test_df["f1_macro"] * 100).tolist()

x = list(range(len(model_names)))
model_colors = {
    "MultinomialNB-NP": "#2980B9",
    "LogisticRegression-NP": "#8E44AD",
    "MLP": "#E67E22",
    "CNN": "#C0392B",
    "AdaBoost (depth=3)": "#27AE60",
    "AdaBoost (LogReg)": "#1F7A5A",
    "AdaBoost (stump)": "#66A182",
    "AdaBoost (GaussianNB)": "#145A32",
}

colors = [model_colors[model] for model in model_names]
fig, axes = plt.subplots(
    1, 2,
    figsize=(max(14, len(model_names) * 1.8), 7),
    sharey=True,
)

charts = [
    (axes[0], acc_scores, "Accuracy trên tập Test (MNIST)", "Accuracy (%)"),
    (axes[1], f1_scores, "Macro-F1 trên tập Test (MNIST)", "Macro-F1 (%)"),
]

for ax, scores, title, ylabel in charts:
    bars = ax.bar(
        x,
        scores,
        color=colors,
        edgecolor="black",
        linewidth=0.7,
        width=0.72,
    )

    for bar, score in zip(bars, scores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{score:.2f}%",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 110)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=35, ha="right", fontsize=9)

    for label, color in zip(ax.get_xticklabels(), colors):
        label.set_color(color)
        label.set_fontweight("bold")

    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.set_axisbelow(True)

fig.suptitle(
    "So sánh Accuracy và Macro-F1 theo mô hình",
    fontsize=14,
    fontweight="bold",
)
fig.tight_layout(rect=(0, 0, 1, 0.94))

chart1_path = FIGURES_DIR / "Comp_01_accuracy_f1_comparison.png"
fig.savefig(chart1_path, dpi=160, facecolor="white")
display(fig)
plt.close(fig)

print(f"Đã tạo và lưu biểu đồ tại: {chart1_path}")

<Figure size 1440x700 with 2 Axes>

Đã tạo và lưu biểu đồ tại: C:\Users\Lenovo\Downloads\Pattern-Recognition_1\figures\Comp_01_accuracy_f1_comparison.png



2. `Comp_02_time_vs_accuracy_pareto.png`: Biểu đồ phân tán đánh đổi Thời gian huấn luyện vs Độ chính xác kèm Đường biên Pareto.

In [54]:
model_colors = {
    "MultinomialNB-NP": "#2980B9",
    "LogisticRegression-NP": "#8E44AD",
    "MLP": "#E67E22",
    "CNN": "#C0392B",
    "AdaBoost (depth=3)": "#27AE60",
    "AdaBoost (LogReg)": "#1F7A5A",
    "AdaBoost (stump)": "#66A182",
    "AdaBoost (GaussianNB)": "#145A32",
}

marker_dict = {
    "MultinomialNB-NP": "s",
    "LogisticRegression-NP": "D",
    "MLP": "^",
    "CNN": "*",
    "AdaBoost (depth=3)": "o",
    "AdaBoost (LogReg)": "p",
    "AdaBoost (stump)": "h",
    "AdaBoost (GaussianNB)": "X",
}

fig, ax = plt.subplots(figsize=(11, 7))

for _, row in test_df.iterrows():
    model = str(row["model"])
    color = model_colors[model]
    marker = marker_dict[model]
    train_time = row["train_seconds"]
    accuracy = row["accuracy"] * 100

    size = 260 if model in ["CNN", "MLP"] else 180
    ax.scatter(
        train_time,
        accuracy,
        color=color,
        marker=marker,
        s=size,
        edgecolors="white",
        linewidth=1.2,
        zorder=5,
    )

    offset_y, offset_x = 0.5, 1.2
    if model == "CNN":
        offset_y, offset_x = -0.8, 1.15
    elif model == "MLP":
        offset_y, offset_x = 0.6, 0.65

    ax.annotate(
        f"{model} ({accuracy:.1f}%)",
        (train_time, accuracy),
        xytext=(train_time * offset_x, accuracy + offset_y),
        fontsize=9,
        fontweight="medium",
        color="#2C3E50",
    )

# Vẽ đường biên Pareto
sorted_by_time = test_df.sort_values("train_seconds").reset_index(drop=True)
pareto_points = []
current_max_accuracy = -1.0

for _, row in sorted_by_time.iterrows():
    if row["accuracy"] > current_max_accuracy:
        pareto_points.append((row["train_seconds"], row["accuracy"] * 100))
        current_max_accuracy = row["accuracy"]

if len(pareto_points) > 1:
    px, py = zip(*pareto_points)
    ax.plot(
        px,
        py,
        linestyle="--",
        color="#34495E",
        linewidth=2,
        alpha=0.9,
        label="Đường biên hiệu quả Pareto",
        zorder=4,
    )

ax.set_xscale("log")
ax.set_xlabel(
    "Thời gian huấn luyện (giây) - Thang logarit",
    fontsize=11,
    fontweight="bold",
)
ax.set_ylabel(
    "Độ chính xác trên tập Test (%)",
    fontsize=11,
    fontweight="bold",
)
ax.set_title(
    "Đánh đổi giữa Thời gian huấn luyện và Độ chính xác (Pareto Trade-off)",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.grid(True, which="both", linestyle="--", alpha=0.3)
ax.set_axisbelow(True)
ax.set_ylim(50, 102)
ax.legend(loc="lower right", fontsize=10, frameon=True)

plt.tight_layout()
chart2_path = FIGURES_DIR / "Comp_02_time_vs_accuracy_pareto.png"
fig.savefig(chart2_path, dpi=160, bbox_inches="tight", facecolor="white")
display(fig)
plt.close(fig)

print(f"Đã tạo và lưu biểu đồ phân tán Pareto tại: {chart2_path}")

<Figure size 1100x700 with 1 Axes>

Đã tạo và lưu biểu đồ phân tán Pareto tại: C:\Users\Lenovo\Downloads\Pattern-Recognition_1\figures\Comp_02_time_vs_accuracy_pareto.png



3. `Comp_03_radar_multi_criteria.png`: Biểu đồ Radar đa tiêu chí đánh giá toàn diện 5 khía cạnh cốt lõi cho các mô hình đại diện.


In [55]:
target_radar_models = [
    "MultinomialNB-NP",
    "LogisticRegression-NP",
    "MLP",
    "CNN",
    "AdaBoost (depth=3)",
]

radar_df = (
    test_df[test_df["model"].isin(target_radar_models)]
    .set_index("model")
    .reindex(target_radar_models)
)

if radar_df.isna().any().any():
    raise ValueError("Thiếu dữ liệu của một hoặc nhiều mô hình được chọn.")

categories = [
    "Accuracy",
    "Macro-F1",
    "Precision",
    "Recall",
    "Tốc độ dự đoán",
]
metric_columns = [
    "accuracy",
    "f1_macro",
    "precision_macro",
    "recall_macro",
]
pi = np.pi
num_vars = len(categories)
angles = [n / num_vars * 2 * pi for n in range(num_vars)]
angles += angles[:1]

model_colors = {
    "MultinomialNB-NP": "#2980b9",
    "LogisticRegression-NP": "#8e44ad",
    "MLP": "#e67e22",
    "CNN": "#c0392c",
    "AdaBoost (depth=3)": "#27ae60",
}

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})

for model in target_radar_models:
    row = radar_df.loc[model]

    # Điểm tốc độ cao hơn nghĩa là thời gian dự đoán ngắn hơn.
    speed_score = 1 / (1 + np.log1p(float(row["predict_seconds"])))
    values = [float(row[column]) for column in metric_columns]
    values.append(speed_score)
    values += values[:1]

    ax.plot(
        angles,
        values,
        linewidth=2,
        marker="o",
        markersize=4,
        label=model,
        color=model_colors[model],
    )
    ax.fill(angles, values, alpha=0.07, color=model_colors[model])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight="bold")
ax.set_ylim(0.3, 1.0)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.4", "0.6", "0.8", "1.0"], fontsize=8, color="gray")
ax.grid(linestyle="--", alpha=0.5)

ax.set_title(
    "So sánh các mô hình trên tập Test",
    fontsize=13,
    fontweight="bold",
    pad=22,
)
ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.08), fontsize=9)

fig.tight_layout()
chart3_path = FIGURES_DIR / "Comp_03_radar_multi_criteria.png"
fig.savefig(chart3_path, dpi=150, bbox_inches="tight")
display(fig)
plt.close(fig)

print(f"Đã tạo và lưu biểu đồ Radar tại: {chart3_path}")

<Figure size 800x800 with 1 Axes>

Đã tạo và lưu biểu đồ Radar tại: C:\Users\Lenovo\Downloads\Pattern-Recognition_1\figures\Comp_03_radar_multi_criteria.png



---
## **4.Kết luận và xếp hạng**

### **4.1. Xếp hạng dựa trên các tiêu chí**



#### **4.1.1. Mô hình nào đạt độ chính xác cao nhất?**
* **Mạng nơ-ron tích chập (`CNN`)** là mô hình có độ chính xác cao nhất tuyệt đối, đạt **98.76%** (chỉ phân loại sai 124 mẫu trên tổng số 10.000 mẫu kiểm thử), đi kèm điểm số Macro-F1 đạt **98.76%**.
* *Lý do vượt trội:* Khác với các phương pháp duỗi phẳng bức ảnh thành vector 1D, CNN bảo toàn hoàn toàn cấu trúc không gian 2 chiều ($28 \times 28$). Tầng tích chập với các bộ lọc cục bộ và cơ chế chia sẻ trọng số cho phép mạng học được các đặc trưng hình học cơ bản của nét chữ (đoạn thẳng, góc cong, điểm khép kín). Tầng Max-Pooling tạo ra tính bất biến đối với các phép dịch chuyển và co giãn nhẹ của chữ viết tay.
* Mô hình đứng thứ hai là `MLP` với độ chính xác **98.26%**, thua kém CNN 0.50% do việc kết nối toàn phần không có tính bất biến không gian và dễ bị ảnh hưởng khi nét chữ bị dịch chuyển vị trí.



#### **4.1.2. Mô hình nào hiệu quả nhất về thời gian huấn luyện?**
* **Xét về tốc độ thuần túy:** `MultinomialNB-NP` là mô hình nhanh nhất vô địch khi chỉ mất **0.06 giây** để huấn luyện trên 50.000 mẫu, nhanh gấp hơn 700 lần so với CNN.
* **Xét về tỷ suất hiệu năng trên chi phí (Hiệu quả tối ưu Pareto):** `MLP` là mô hình xuất sắc nhất. Chỉ mất **30.27 giây** huấn luyện trên CPU thông thường, MLP đã đạt tới **98.26%** độ chính xác.
* **Mô hình kém hiệu quả nhất:** `AdaBoost (depth=3)`. Tiêu tốn tới **11.019.24 giây** (hơn 3 giờ đồng hồ) nhưng chỉ đạt 91.75% độ chính xác. Tỷ lệ đánh đổi thời gian lấy độ chính xác của AdaBoost hoàn toàn không tương xứng khi so với các giải pháp mạng nơ-ron.



#### **4.1.3. Mô hình nào phù hợp nhất để triển khai thực tế (Real-time Deployment)?**
* **Đối với các thiết bị có tài nguyên phần cứng hạn chế (CPU yếu, bộ nhớ RAM thấp):**
  * `MLP` và `LogisticRegression-NP` là sự lựa chọn hàng đầu. Cả hai mô hình có độ trễ suy diễn cực thấp: **2.3 µs/mẫu** đối với MLP và **2.6 µs/mẫu** đối với Logistic Regression. Hệ thống có thể xử lý hơn 380.000 bức ảnh mỗi giây trên CPU đơn nhân tiêu chuẩn mà không cần bất kỳ bộ tăng tốc phần cứng nào.
* **Đối với các hệ thống nhận dạng yêu cầu độ chính xác tối đa:**
  * `CNN` là lựa chọn chuẩn mực. Với độ trễ **41.3 µs/mẫu** (~24.200 ảnh/giây), CNN hoàn toàn đáp ứng trơn tru các ứng dụng đọc số tài khoản ngân hàng, mã bưu chính hay số hóa biểu mẫu theo thời gian thực.
* **Lưu ý:** Các mô hình Boosting (`AdaBoost`) không phù hợp cho các dịch vụ đòi hỏi độ trễ thấp do cấu trúc phân loại tuần tự làm tăng độ trễ lên tới 266 - 784 µs cho mỗi mẫu.

### **4.2. Bảng tổng hợp xếp hạng**

| Hạng | Mô hình | Accuracy | Train (s) | Độ trễ / mẫu | Ưu điểm chính | Hạn chế chính | Khuyến nghị áp dụng |
| :---: | :--- | :---: | :---: | :---: | :--- | :--- | :--- |
| **1** | **CNN** | **98.76%** | 43.6s | 41.3 µs | Chính xác nhất, giữ đặc trưng không gian 2D | Tốn tài nguyên hơn MLP | Ứng dụng thực tế cần độ chính xác tối đa |
| **2** | **MLP** | **98.26%** | 30.3s | **2.3 µs** | Suy diễn siêu tốc, độ chính xác rất cao | Thiếu tính bất biến vị trí | Thiết bị nhúng, IoT, luồng ảnh thời gian thực |
| **3** | **AdaBoost (depth=3)** | **91.75%** | 11019s | 266.8 µs | Bắt được tương tác phi tuyến bậc thấp | Train quá lâu (~3h), suy diễn chậm | Nghiên cứu lý thuyết học kết hợp |
| **4** | **Logistic Regression** | **90.18%** | 32.3s | 2.6 µs | Cực kỳ đơn giản, suy diễn nhanh | Chỉ phân tách được ranh giới tuyến tính | Baseline chuẩn cho phân loại tuyến tính |
| **5** | **AdaBoost (LogReg)** | **85.06%** | 48.4s | 30.1 µs | Train nhanh hơn cây quyết định | Kém hiệu quả hơn Logistic đơn lẻ | Không khuyến nghị sử dụng |
| **6** | **MultinomialNB** | **83.56%** | **0.06s** | 33.3 µs | Train tức thì (chỉ cần đếm tần suất) | Giả định độc lập sai lệch với dữ liệu ảnh | Thử nghiệm nhanh dữ liệu (Proof-of-Concept) |
| **7** | **AdaBoost (stump)** | **78.53%** | 242.0s | 242.0 µs | Cấu trúc cực đơn giản (cắt theo 1 pixel) | Quá yếu cho bài toán 10 chữ số | Minh họa bài giảng lý thuyết Boosting |
| **8** | **AdaBoost (GaussianNB)** | **56.41%** | 46.8s | 784.1 µs | - | Phân phối chuẩn xung đột với ảnh nhị phân | Không sử dụng |